# 第 2 節｜YOLO11n 物件偵測、互動標註與微調

**第 2 節｜預訓練偵測 → 下載 COCO8 → 在 Notebook 畫框 → 儲存標註 → 微調 → 使用新圖片預測。**

使用 **Google Colab T4 GPU**；將教學 ZIP 解壓縮到 `/content/`。若以 VS Code Colab 擴充功能連線，請確定工作目錄有 `04_素材/` 和 `04_人工標註練習.csv`。完整微調時間可能超出 20 分鐘，建議課前先下載權重及資料。

## 1. 安裝套件、匯入模組、使用預訓練模型

**套件安裝與 Python `import` 分成兩格。** 使用官方的 `yolo11n.pt`，可以不訓練先辨認 COCO 常見類別。

In [ ]:
# Install the pretrained YOLO package and interactive box editor.
# 安裝 YOLO 與可直接在 Notebook 畫框的互動標註工具。
%pip -q install "ultralytics>=8.3,<9" "jupyter-bbox-widget==0.7.0"

In [ ]:
# Import helpers for file paths.
# 匯入檔案路徑工具。
from pathlib import Path

# Import helpers for copying images.
# 匯入複製圖片的工具。
import shutil

# Import OpenCV for converting and displaying images.
# 匯入 OpenCV 處理與顯示影像。
import cv2

# Import Pandas for reading and saving annotations.
# 匯入 Pandas 讀取與儲存標註資料。
import pandas as pd

# Import PyTorch to check GPU availability.
# 匯入 PyTorch 檢查是否有 GPU。
import torch

# Import Pillow for image dimensions and previews.
# 匯入 Pillow 取得圖片尺寸並預覽。
from PIL import Image

# Import Notebook output rendering tools.
# 匯入 Notebook 輸出顯示工具。
from IPython.display import display

# Import YOLO for model inference and training.
# 匯入 YOLO 模型執行推論及微調。
from ultralytics import YOLO

# Import the official COCO8 dataset checker.
# 匯入 COCO8 資料集下載與檢查工具。
from ultralytics.data.utils import check_det_dataset

# Import the image-annotation widget.
# 匯入可以用滑鼠直接畫框的標註工具。
from jupyter_bbox_widget import BBoxWidget

# Enable the custom widget manager when using Colab.
# 使用 Colab 網頁版時啟用第三方互動元件。
try:
    # Import Colab's widget support if available.
    # 如果在 Colab 環境中，就載入互動元件支援。
    from google.colab import output

    # Enable rendering of the interactive annotation editor.
    # 讓 Notebook 能夠顯示互動標註工具。
    output.enable_custom_widget_manager()
except ImportError:
    # Outside Colab, the normal Jupyter widget renderer is used.
    # 非 Colab 環境使用 Jupyter 原有的互動元件機制。
    pass

In [ ]:
# Confirm whether the remote runtime has a GPU.
# 檢查 Colab 是否有配置 GPU。
print("GPU：", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# Load COCO-pretrained YOLO11 Nano weights.
# 載入已在 COCO 上預訓練的 YOLO11 Nano。
model = YOLO("yolo11n.pt")

In [ ]:
# Predict objects in the example cat photograph.
# 讓預訓練 YOLO 偵測貓咪照片中的物件。
result = model.predict("04_素材/cat.jpg", conf=0.25, verbose=False)[0]

# Display the predicted boxes and class names.
# 顯示模型預測的類別名稱與辨識框。
display(Image.fromarray(cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB)))

## 2. 自動下載 COCO8 示範資料集

官方 COCO8 包含 **4 張 Train、4 張 Validation**。這只是幫助學生確認訓練流程，資料量不足以保證微調效果。額外人工標註只加入 **Train**，不更動原本的 Validation。

In [ ]:
# Download COCO8 and locate its training images.
# 下載 COCO8，取得已標註的訓練照片位置。
dataset = check_det_dataset("coco8.yaml")
train_images = Path(dataset["train"])

# Inspect the number of training images.
# 顯示訓練影像的數量。
print("COCO8 Train 圖片數：", len(list(train_images.glob("*.jpg"))))

## 3. 直接在 Notebook 標註物件

本節改用 [jupyter-bbox-widget](https://github.com/gereleth/jupyter-bbox-widget)：可直接**滑鼠拖曳新增框、移動、改變大小、刪除框及選擇類別**。

先將 `IMAGE_NAME` 設成 `cat.jpg`、`coffee.jpg` 或 `motorcycle.jpg`（之後也可自行放入新照片）。它會載入 CSV 內該圖片的既有框，供修改。類別名稱只示範三種：**cat = 15、cup = 41、motorcycle = 3**（COCO 的編號）。這些是模型原本就認識的類別；新增自己的全新類別，需要另行設計資料集類別表。

In [ ]:
# Choose the photo to annotate directly in the notebook.
# 選擇要在 Notebook 裡直接用滑鼠標註的照片。
IMAGE_NAME = "cat.jpg"

# Read existing boxes so students can edit instead of starting over.
# 讀取已有的框，讓學生可以直接修改位置。
annotations = pd.read_csv("04_人工標註練習.csv")
image_path = Path("04_素材") / IMAGE_NAME
image_width, image_height = Image.open(image_path).size

# Name the three COCO classes used in the lesson.
# 指定本課會框選的三種 COCO 類別。
class_names = {3: "motorcycle", 15: "cat", 41: "cup"}
boxes = []

# Convert saved YOLO coordinates back to image pixels.
# 將 CSV 的 YOLO 相對座標還原為圖片像素位置。
for row in annotations[annotations.image == IMAGE_NAME].itertuples():
    # Keep one rectangle and its class name.
    # 把單個矩形及類別存入編輯工具的格式。
    boxes.append({"x": round((row.cx - row.w / 2) * image_width),
                  "y": round((row.cy - row.h / 2) * image_height),
                  "width": round(row.w * image_width),
                  "height": round(row.h * image_height),
                  "label": class_names[int(row.class_id)]})

In [ ]:
# Create an editor with the saved boxes already visible.
# 建立可直接拖曳的標註工具，並顯示原有的框。
editor = BBoxWidget(image=str(image_path), classes=list(class_names.values()), bboxes=boxes, hide_buttons=True)

# Show the interactive canvas below this cell.
# 在這格下方直接顯示滑鼠畫框介面。
display(editor)

### 儲存剛剛畫好的標註

**畫完或修改完後，再執行下一格。** 它會覆蓋 CSV 裡「目前這張圖片」的舊框，保留其他圖片的紀錄。每張可畫 **1 個或多個框**。標註座標會自動從像素轉為 YOLO 要的 0～1 相對座標。無須自己輸入小數。

In [ ]:
# Match selected label names to COCO class IDs.
# 把學生選的類別名稱對應到 COCO 編號。
class_ids = {"motorcycle": 3, "cat": 15, "cup": 41}
rows = []

# Convert each drawn pixel box into normalized YOLO coordinates.
# 將每個滑鼠畫出的框轉成 YOLO 相對座標。
for box in editor.bboxes:
    # Compute the normalized center and size.
    # 計算框中心、寬度及高度的 0～1 比例。
    cx = (box["x"] + box["width"] / 2) / image_width
    cy = (box["y"] + box["height"] / 2) / image_height
    w = box["width"] / image_width
    h = box["height"] / image_height

    # Keep all boxes, including multiple objects in one photo.
    # 將同一張照片的多個框完整保留下來。
    rows.append([IMAGE_NAME, class_ids[box["label"]], cx, cy, w, h])

In [ ]:
# Prevent accidentally deleting all labels of this photo.
# 避免忘記畫框時，意外清空這張照片的標註。
if not rows:
    raise ValueError("請先在上一格的圖片畫至少一個框，再執行儲存。")

# Replace only the selected image's annotations.
# 只更新目前這張圖片，其它照片的標註保持不變。
new_rows = pd.DataFrame(rows, columns=annotations.columns)
annotations = pd.concat([annotations[annotations.image != IMAGE_NAME], new_rows], ignore_index=True)

# Save the annotation CSV for future classes or sessions.
# 將框的位置存成 CSV，方便下次繼續使用。
annotations.to_csv("04_人工標註練習.csv", index=False, encoding="utf-8-sig")
display(annotations)

**繼續標註其他圖片：**把 `IMAGE_NAME` 改成 `coffee.jpg` 或 `motorcycle.jpg`，重新執行「選圖片 → 畫框 → 儲存」三個步驟。

**介面未顯示時：**先確認前面已執行安裝和 `import`，再使用 Colab 網頁版重新開啟 Notebook；部分 VS Code／Colab 前端可能不支援第三方互動元件。原本的 CSV 仍可直接修改作為備用方式，不會阻止訓練。

若互動工具顯示空白，安裝完成後可**重新啟動執行階段**再執行；不同版本的 Colab 網頁和 VS Code 互動元件支援可能不一致。

## 4. 將儲存的框轉為 YOLO 訓練標註

CSV 方便人閱讀；YOLO 訓練需要**每張照片對應一個同名 `.txt`**，每個物件一行 `class_id cx cy w h`。以下會自動複製原始照片、建立標註檔，不需手動計算。

In [ ]:
# Reload the saved CSV, including all images annotated so far.
# 重新讀取剛才儲存的所有圖片標註。
annotations = pd.read_csv("04_人工標註練習.csv")

# Find the matching training label folder.
# 找到 COCO8 訓練圖片對應的標註資料夾。
train_labels = Path(str(train_images).replace("/images/", "/labels/"))
train_labels.mkdir(parents=True, exist_ok=True)

# Copy each labeled image without changing the original.
# 每張已標註圖片只複製一次，不變更教材原檔。
for filename in annotations.image.unique():
    shutil.copy2(Path("04_素材") / filename, train_images / filename)

# Save one YOLO text file per image, allowing multiple boxes.
# 每張照片存一個 YOLO 標註檔，可包含多個物件框。
for filename, group in annotations.groupby("image"):
    label_path = train_labels / Path(filename).with_suffix(".txt").name
    label_path.write_text(group[["class_id", "cx", "cy", "w", "h"]].to_csv(sep=" ", header=False, index=False))

In [ ]:
# Show the training set size and one label file.
# 確認圖片與標註已加入 YOLO 訓練資料。
print("Train 圖片數：", len(list(train_images.glob("*.jpg"))))

# Print the first image's YOLO text labels.
# 顯示第一張照片的 YOLO 標註文字。
print((train_labels / Path(annotations.iloc[0].image).with_suffix(".txt").name).read_text())

## 5. 使用 T4 微調 YOLO11n

修改 `EPOCHS`（訓練輪數）與 `BATCH_SIZE`（每批圖片數），其餘先使用預設值。示範採用極小資料集，主要目的在走過**標註 → 模型訓練 → 模型預測**，不能當作可靠的效能提升實驗。

In [ ]:
# Choose how many times to pass through the dataset.
# 設定訓練資料要重複學習幾個 Epoch。
EPOCHS = 5

# Choose the number of images per training batch.
# 設定一次訓練要處理幾張圖片。
BATCH_SIZE = 4

# Select the GPU when Colab provides one.
# 有 T4 時使用 GPU，沒有則退回 CPU。
DEVICE = 0 if torch.cuda.is_available() else "cpu"

# Fine-tune pretrained YOLO11n with our added annotations.
# 使用包含自己標註照片的 COCO8 訓練資料微調 YOLO11n。
model = YOLO("yolo11n.pt")
model.train(data="coco8.yaml", epochs=EPOCHS, batch=BATCH_SIZE, imgsz=640, device=DEVICE, project="04_runs", name="with_labels", exist_ok=True)

## 6. 查看 Loss 曲線與最佳模型

`results.png` 是實際訓練產生的 Loss／驗證圖表；`best.pt` 是 Ultralytics 根據驗證表現選出的權重，**不一定是最後一個 Epoch**。

In [ ]:
# Locate the checkpoint selected by Ultralytics validation.
# 找出 YOLO 依驗證指標挑選的最佳權重。
best_path = Path("04_runs/with_labels/weights/best.pt")

# Display the checkpoint path and training plots.
# 顯示權重位置和訓練、驗證曲線。
print("最佳模型：", best_path.resolve())
display(Image.open("04_runs/with_labels/results.png"))

## 7. 使用模型預測新圖片

先觀察訓練照片，再換用**未放入 Train／Validation 的新照片**，例如 `dog.jpg`、`street_detection.jpg`。與預訓練模型相比，少量資料微調後不保證更準。

In [ ]:
# Load the best fine-tuned checkpoint.
# 載入微調後表現最佳的 YOLO 模型。
fine = YOLO(str(best_path))

# Predict the three photos we annotated in class.
# 檢視剛才標註過的三張教材照片。
results = fine.predict(["04_素材/cat.jpg", "04_素材/coffee.jpg", "04_素材/motorcycle.jpg"], conf=0.25, verbose=False)

# Display predicted bounding boxes for each image.
# 顯示每張圖片的預測框與類別。
for result in results:
    display(Image.fromarray(cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB)))

In [ ]:
# Choose a picture that was not used for training.
# 換成一張沒有加入訓練集的新照片。
new_image = "04_素材/dog.jpg"

# Run the best fine-tuned model on the new image.
# 使用微調後最佳模型預測新照片。
new_result = fine.predict(new_image, conf=0.25, verbose=False)[0]

# Display the final predictions on the new image.
# 顯示新圖片的類別與偵測框。
display(Image.fromarray(cv2.cvtColor(new_result.plot(), cv2.COLOR_BGR2RGB)))

**延伸練習：**改成 `04_素材/street_detection.jpg`，比較結果是否不同。也可以自行把其他 JPG 放進 `04_素材/`，修改 `IMAGE_NAME`，重新標註、儲存、微調。

**注意：**目前資料極少，不足以證明泛化能力，也不應把已加入訓練的照片當成獨立 Test。若拍攝圖片包含可識別的車牌或人物，請先取得授權並注意隱私。